# SASRec serving correctness + precision audit

Before accepting any latency number, this notebook checks the **actual trained ML-1M SASRec checkpoint** against its proposed KV-cache serving path.

It reports:
- canonical full-catalog NDCG@10 in FP32 and BF16 inference;
- exact token-by-token KV-cache NDCG/equivalence while a prefix grows inside the trained 200-token window;
- the critical **sliding-window test** after 200 tokens (learned absolute positions make old cached K/V potentially stale);
- full-window recompute vs cached-append latency, separately for FP32 and BF16;
- Walker FP32/BF16 NDCG as a precision sanity check;
- a fairness manifest showing the historical training-precision mismatch.

Dense catalog scoring is used only as a quality oracle here; it is not part of the Walker timed serving path.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, sys, shutil, subprocess, torch
REPO='/content/Sparsewalker'
BRANCH='agent/serving-speed-benchmark'
if os.path.exists(REPO): shutil.rmtree(REPO)
subprocess.run(['git','clone','-q','-b',BRANCH,'https://github.com/hanialshater/Sparsewalker-.git',REPO],check=True)
sys.path.insert(0,f'{REPO}/src')
sys.path.insert(0,f'{REPO}/experiments')
for name in list(sys.modules):
    if name=='sparsewalker' or name.startswith('sparsewalker.'):
        del sys.modules[name]
assert torch.cuda.is_available(), 'GPU runtime required'
print('GPU',torch.cuda.get_device_name(0),'bf16',torch.cuda.is_bf16_supported())
print('BRANCH',BRANCH)


## Run audit

The key outputs to paste back are `FAIRNESS`, `SASREC_NDCG`, `INCREMENTAL_EQUIVALENCE_FP32`, `SLIDING_WINDOW_FP32`, `LATENCY_FP32`, `LATENCY_BF16`, `WALKER_NDCG`, and `DECISION`.

In [ ]:
import runpy, sys
SCRIPT=f'{REPO}/benchmarks/run_sasrec_serving_correctness_audit.py'
sys.argv=[SCRIPT,'--incremental-users','1024','--sliding-users','512']
print('SASREC SERVING AUDIT START',flush=True)
runpy.run_path(SCRIPT,run_name='__main__')
print('SASREC SERVING AUDIT END',flush=True)


## Compact saved result

In [ ]:
import json
from pathlib import Path
p=Path('/content/drive/MyDrive/sparsewalker_speed/sasrec_serving_correctness_audit.json')
if p.exists():
    r=json.loads(p.read_text())
    for k in ['fairness','sasrec_full_quality','incremental_fp32','sliding_fp32','latency_fp32','latency_bf16','walker_quality','decision']:
        print('\n'+k.upper(),json.dumps(r.get(k),indent=2))
